In [0]:
# --------------------------------------------------
# 1. GET CURRENT RUN
# --------------------------------------------------

running_runs = spark.sql("""
    SELECT
        run_id,
        batch_id
    FROM workspace.control.etl_run_log
    WHERE pipeline_name = 'orders_pipeline'
      AND status = 'RUNNING'
""").collect()

if len(running_runs) != 1:
    raise ValueError(
        f"Expected exactly 1 RUNNING run, found {len(running_runs)}"
    )

run_id = running_runs[0]["run_id"]
batch_id = running_runs[0]["batch_id"]

print(f"Run ID:   {run_id}")
print(f"Batch ID: {batch_id}")


# --------------------------------------------------
# 2. PROCESS BRONZE
# --------------------------------------------------

try:

    # Count before load
    bronze_before = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM workspace.bronze.orders
        WHERE batch_id = '{batch_id}'
    """).first()["cnt"]


    spark.sql(f"""
    INSERT INTO workspace.bronze.orders

    SELECT
         l.order_id,
        l.customer_id,
        l.amount,
        l.status,
        l.order_date,
        l.last_updated,
        l.event_id,
        l.batch_id,
        l.source_file,
        l.load_timestamp,
        l.currency,
        l.payment_method


    FROM workspace.landing.orders l

    WHERE l.batch_id = '{batch_id}'

      AND NOT EXISTS (
          SELECT 1
          FROM workspace.bronze.orders b
          WHERE b.batch_id = l.batch_id
            AND b.event_id = l.event_id
      )
    """)

    print("Bronze load completed.")


    # --------------------------------------------------
    # 3. COUNT INSERTED ROWS THIS RUN
    # --------------------------------------------------

    bronze_after = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM workspace.bronze.orders
        WHERE batch_id = '{batch_id}'
    """).first()["cnt"]

    bronze_rows = bronze_after - bronze_before

    print(f"Bronze inserted this run: {bronze_rows}")


    # --------------------------------------------------
    # 4. UPDATE AUDIT
    # --------------------------------------------------

    spark.sql(f"""
        UPDATE workspace.control.etl_run_log

        SET bronze_rows = {bronze_rows}

        WHERE run_id = '{run_id}'
          AND status = 'RUNNING'
    """)

    print("Audit updated successfully.")


    # --------------------------------------------------
    # 5. SUMMARY
    # --------------------------------------------------

    print("----------------------------------")
    print("BRONZE PROCESSING COMPLETE")
    print("----------------------------------")
    print(f"Run ID:                  {run_id}")
    print(f"Batch ID:                {batch_id}")
    print(f"Bronze inserted this run:{bronze_rows}")
    print("----------------------------------")


except Exception as e:

    error_message = str(e).replace("'", "''")[:4000]

    spark.sql(f"""
        UPDATE workspace.control.etl_run_log

        SET
            end_timestamp = CURRENT_TIMESTAMP(),
            status = 'FAILED',
            error_message = '{error_message}'

        WHERE run_id = '{run_id}'
          AND status = 'RUNNING'
    """)

    print("Bronze processing FAILED.")
    print(error_message)

    raise

In [0]:
%sql
SELECT
    run_id,
    batch_id,
    start_timestamp,
    end_timestamp,
    bronze_rows,
    status,
    error_message
FROM workspace.control.etl_run_log
ORDER BY start_timestamp DESC;

In [0]:
%sql
select * from control.etl_run_log
order by start_timestamp desc
limit 1